In [1]:
import os, sys

PROJECT_ROOT = '/scratch/jq2uw/derm_vlms'
MEDGEMMA_DIR = os.path.join(PROJECT_ROOT, 'collect_ai_response', 'medgemma')

if MEDGEMMA_DIR not in sys.path:
    sys.path.insert(0, MEDGEMMA_DIR)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

import torch
torch.cuda.empty_cache()

# HF token for gated MedGemma model
sys.path.insert(0, PROJECT_ROOT)
from tokens import HF_TOKEN

from utils import load_model, predict_image, parse_label

print('Loading model...')
model, processor = load_model(hf_token=HF_TOKEN)
print('Model loaded.')


/home/jq2uw/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.60s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Total params: 4,300,079,472
Model loaded.


In [2]:
import pandas as pd
from pathlib import Path

sys.path.insert(0, PROJECT_ROOT)
from data_utils import prepare_all_lesions

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
IMAGES_DIR = os.path.join(RESULTS_DIR, 'images')
OUT_PATH = os.path.join(RESULTS_DIR, 'medgemma_predictions_all.csv')

df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data_share', 'midas_share.parquet'))
print(f'Loaded {len(df)} rows')
print(f'y3 distribution:\n{df["y3"].value_counts()}')
print(f'y16 distribution:\n{df["y16"].value_counts()}')

df_all = prepare_all_lesions(df, data_dir=DATA_DIR, output_dir=IMAGES_DIR)
print(f'\nTotal prediction rows: {len(df_all)}')
df_all.head(10)


Loaded 3357 rows
y3 distribution:
y3
malignant    1391
benign       1322
other         644
Name: count, dtype: int64
Sampled 10 lesions (5 per class) -> 30 rows
Classes: {'benign': 15, 'malignant': 15}
Images saved to: /scratch/jq2uw/derm_vlms/results/images


,id,ground_truth,image_mode,image_path,original_image_name,lesion_id
0,1_photo,benign,photo,/scratch/jq2uw/derm_vlms/results/images/1_phot...,s-prd-667118134.jpg,534_left lower leg_no
1,1_dscope,benign,dscope,/scratch/jq2uw/derm_vlms/results/images/1_dsco...,s-prd-667118139.jpg,534_left lower leg_no
2,1_combined,benign,combined,/scratch/jq2uw/derm_vlms/results/images/1_comb...,s-prd-667118134.jpg; s-prd-667118139.jpg,534_left lower leg_no
3,2_photo,benign,photo,/scratch/jq2uw/derm_vlms/results/images/2_phot...,s-prd-505211076.jpg,215_central chest _no
4,2_dscope,benign,dscope,/scratch/jq2uw/derm_vlms/results/images/2_dsco...,s-prd-505211307.jpg,215_central chest _no


In [3]:
from PIL import Image
from tqdm import tqdm

PROMPT = ("You are an expert dermatologist. Please describe using dermatological "
          "terms to describe the appearance of this lesion. "
          "Please give the top 3 diagnoses in your differential.")

CHECKPOINT_EVERY = 10
COL_ORDER = ['id', 'ground_truth', 'y16', 'y16_description', 'image_mode',
             'describe_then_classify', 'image_path', 'original_image_name', 'lesion_id']

# --- Resume logic ---
if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH)
    done_ids = set(existing_df['id'].tolist())
    print(f'Resuming: {len(done_ids)} predictions already completed')
else:
    done_ids = set()

pending = df_all[~df_all['id'].isin(done_ids)]
print(f'{len(pending)} predictions remaining out of {len(df_all)} total')

# --- Prediction loop with checkpointing ---
batch = []
lesions_in_batch = set()

for _, row in tqdm(pending.iterrows(), total=len(pending)):
    try:
        image = Image.open(row['image_path']).convert('RGB')
    except Exception as e:
        print(f'[SKIP] {row["id"]}: {e}')
        continue

    batch.append({
        'id': row['id'],
        'ground_truth': row['ground_truth'],
        'y16': row['y16'],
        'y16_description': row['y16_description'],
        'image_mode': row['image_mode'],
        'describe_then_classify': predict_image(model, processor, image, prompt=PROMPT),
        'image_path': row['image_path'],
        'original_image_name': row['original_image_name'],
        'lesion_id': row['lesion_id'],
    })

    lesions_in_batch.add(row['lesion_id'])

    if len(lesions_in_batch) >= CHECKPOINT_EVERY:
        batch_df = pd.DataFrame(batch)[COL_ORDER]
        header = not os.path.exists(OUT_PATH)
        batch_df.to_csv(OUT_PATH, mode='a', header=header, index=False)
        print(f'\n[Checkpoint] +{len(batch)} rows ({len(lesions_in_batch)} lesions)')
        batch = []
        lesions_in_batch = set()

if batch:
    batch_df = pd.DataFrame(batch)[COL_ORDER]
    header = not os.path.exists(OUT_PATH)
    batch_df.to_csv(OUT_PATH, mode='a', header=header, index=False)
    print(f'\n[Final] +{len(batch)} rows ({len(lesions_in_batch)} lesions)')

print(f'\nAll predictions saved to {OUT_PATH}')


  0%|                                                                                     | 0/30 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
  3%|██▌                                                                          | 1/30 [00:51<24:59, 51.72s/it]Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
  7%|█████▏                                                                       | 2/30 [01:36<22:14, 47.67s/it]Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
 10%|███████▋                                    

Collected 30 predictions


In [4]:
results_df = pd.read_csv(OUT_PATH)
print(f'Total predictions: {len(results_df)}')
print(f'Unique lesions:    {results_df["lesion_id"].nunique()}')
print(f'\nImage mode distribution:\n{results_df["image_mode"].value_counts()}')
print(f'\ny3 distribution:\n{results_df["ground_truth"].value_counts()}')
print(f'\ny16 distribution:\n{results_df["y16"].value_counts()}')
results_df.head(10)


Saved 30 rows to /scratch/jq2uw/derm_vlms/results/medgemma_predictions_paired.csv


,id,ground_truth,image_mode,describe,classify,describe_then_classify,original_image_name,lesion_id
0,1_photo,benign,photo,"Based on the image, here's a description of th...","Based on the image, here are the top 3 potenti...","Based on the image, here's a description of th...",s-prd-667118134.jpg,534_left lower leg_no
1,1_dscope,benign,dscope,"Based on the image, here's a description of th...","Based on the image, the top 3 differential dia...","Based on the image, here's a description of th...",s-prd-667118139.jpg,534_left lower leg_no
2,1_combined,benign,combined,"Based on the provided image, here's a descript...","Based on the image, the top 3 differential dia...","Based on the provided image, here's a descript...",s-prd-667118134.jpg; s-prd-667118139.jpg,534_left lower leg_no
3,2_photo,benign,photo,"Based on the image, here's a description of th...","Based on the image, here are the top 3 potenti...","Based on the image, here's a description of th...",s-prd-505211076.jpg,215_central chest _no
4,2_dscope,benign,dscope,"Based on the image, here's a description of th...","Based on the image, here are the top 3 differe...","Based on the image, here's a description of th...",s-prd-505211307.jpg,215_central chest _no
5,2_combined,benign,combined,"Based on the provided image, here's a descript...","Based on the image, the top 3 differential dia...","Based on the provided image, here's a descript...",s-prd-505211076.jpg; s-prd-505211307.jpg,215_central chest _no
6,3_photo,benign,photo,"Based on the image, here's a description of th...","Based on the image, the top 3 differential dia...","Based on the image, here's a description of th...",s-prd-529717709.jpg,262_left buttock_no
7,3_dscope,benign,dscope,"Based on the image, here's a description of th...","Based on the image, the top 3 differential dia...","Based on the image, here's a description of th...",s-prd-529718156.jpg,262_left buttock_no
8,3_combined,benign,combined,"Based on the provided images, here's a descrip...","Based on the image, the top 3 differential dia...","Based on the provided images, here's a descrip...",s-prd-529717709.jpg; s-prd-529718156.jpg,262_left buttock_no
9,4_photo,benign,photo,"Based on the image, here's a description of th...","Based on the image, the top 3 differential dia...","Based on the image, here's a description of th...",s-prd-649658541.jpg,476_mid vertex_no
